# Periodic-Branch Trigger-Mode Benchmark: LOO Posterior Probability vs Local Bayes Factor

This notebook compares the two dip-triggering choices available inside the MALCA Bayesian event scorer:

- **LOO posterior probability**: per-point leave-one-out posterior probability that the point belongs to the dip/event component.
- **Local Bayes factor**: per-point local log evidence ratio for the event model relative to the baseline model.

The synthetic population is the same family used by `periodic_branch_simulation_benchmark.ipynb`: noisy, seasonal, multi-camera, imperfect periodic light curves with one-off dips injected into most trials and control light curves in the remainder. The default sample size is **12,000 simulated light curves**, which is 1/10 of the 120,000-light-curve default in the periodic-branch benchmark.

The important design choice here is that each light curve and baseline mode is scored once. The notebook then applies both trigger families, including threshold sweeps, to the same `event_probability` and `log_bf_local` arrays. That keeps the comparison focused on trigger behavior rather than differences in simulation draws or baseline fitting.


## How to Run

By default this notebook runs:

- `N_TRIALS = 12000`
- all four periodic-branch modes: true period, 1% selected-period scatter, 5% selected-period scatter, and masked-GP control
- production thresholds plus threshold sweeps for both trigger families

Useful environment overrides before execution:

- `MALCA_TRIGGER_MODE_N_TRIALS=12000` changes the number of simulated light curves.
- `MALCA_TRIGGER_MODE_WORKERS=8` changes process count.
- `MALCA_TRIGGER_MODE_MODES=phase_template_true_period,phase_template_1pct_period_error` limits baseline modes.
- `MALCA_TRIGGER_MODE_FORCE=1` recomputes cached results.
- `MALCA_TRIGGER_MODE_SMOKE=1` runs a quick 48-light-curve smoke test.

Results are cached under `output/diagnostics/trigger_mode_simulation_benchmark/<RUN_TAG>/`.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "malca" / "evaluation").is_dir() and (candidate / "malca" / "events.py").exists():
            return candidate
    raise RuntimeError(f"Could not find MALCA repository root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

from malca.evaluation.trigger_mode_simulation_benchmark import (
    DEFAULT_LOGBF_THRESHOLDS,
    DEFAULT_POSTERIOR_PROBABILITY_THRESHOLDS,
    TriggerModeBenchmarkConfig,
    load_trigger_mode_simulation_benchmark,
    plot_trigger_mode_trial_diagnostic,
    run_trigger_mode_simulation_benchmark,
    select_disagreement_trials,
    summarize_trigger_results,
    trigger_profiles,
)
from malca.triggering import posterior_probability_threshold

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
plt.rcParams["figure.dpi"] = 120


In [ ]:
SMOKE = os.environ.get("MALCA_TRIGGER_MODE_SMOKE", "0") == "1"
N_TRIALS = int(os.environ.get("MALCA_TRIGGER_MODE_N_TRIALS", "48" if SMOKE else "12000"))
WORKERS = int(os.environ.get("MALCA_TRIGGER_MODE_WORKERS", "1" if SMOKE else "8"))
FORCE = os.environ.get("MALCA_TRIGGER_MODE_FORCE", "0") == "1"
RUN_TAG = os.environ.get(
    "MALCA_TRIGGER_MODE_RUN_TAG",
    f"trigger_mode_n{N_TRIALS}_seed20260514" + ("_smoke" if SMOKE else ""),
)
MODE_NAMES = tuple(
    x.strip()
    for x in os.environ.get(
        "MALCA_TRIGGER_MODE_MODES",
        "phase_template_true_period,phase_template_1pct_period_error,phase_template_5pct_period_error,gp_masked_control",
    ).split(",")
    if x.strip()
)
OUTPUT_BASE_DIR = Path(
    os.environ.get(
        "MALCA_TRIGGER_MODE_OUTPUT_DIR",
        str(REPO_ROOT / "output" / "diagnostics" / "trigger_mode_simulation_benchmark"),
    )
)

config = TriggerModeBenchmarkConfig(
    output_base_dir=OUTPUT_BASE_DIR,
    run_tag=RUN_TAG,
    n_trials=N_TRIALS,
    workers=WORKERS,
    force=FORCE,
    show_progress=True,
    mode_names=MODE_NAMES,
    posterior_probability_thresholds=DEFAULT_POSTERIOR_PROBABILITY_THRESHOLDS,
    logbf_thresholds=DEFAULT_LOGBF_THRESHOLDS,
)

print(config)
print("\nTrigger profiles:")
display(pd.DataFrame(trigger_profiles(config)))


## Execute or Load Cached Benchmark

This cell generates the synthetic design, scores each simulated light curve under each selected baseline mode, applies the trigger threshold sweeps, and writes parquet/CSV summaries. If the run directory already exists, it loads cached outputs unless `MALCA_TRIGGER_MODE_FORCE=1` is set.


In [ ]:
run = run_trigger_mode_simulation_benchmark(config)
results = run.trigger_results.copy()
scores = run.score_results.copy()
plots_dir = run.run_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

print("run_dir:", run.run_dir)
print("trial design:", run.trial_design.shape)
print("score rows:", scores.shape)
print("trigger rows:", results.shape)
print("production rows:", results[results["is_production"].fillna(False)].shape)


## Simulation Population Audit

These plots show whether the run spans the intended input space: periods, periodic amplitudes, injected dip amplitudes and widths, camera offsets, and baseline imperfections. Use this first to catch accidental over-narrowing of the simulated population.


In [ ]:
design = run.trial_design.copy()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
axes[0].hist(design["period_days"], bins=np.logspace(np.log10(0.8), np.log10(80), 35), color="tab:blue", alpha=0.8)
axes[0].set_xscale("log")
axes[0].set_title("Injected periodic baseline periods")
axes[0].set_xlabel("period [days]")
axes[1].hist(design["periodic_amp_mag"], bins=35, color="tab:orange", alpha=0.85)
axes[1].set_title("Periodic baseline amplitude")
axes[1].set_xlabel("mag")
axes[2].hist(design.loc[design["has_dip"], "dip_amp_mag"], bins=35, color="tab:green", alpha=0.85)
axes[2].set_title("Injected one-off dip amplitudes")
axes[2].set_xlabel("mag")
axes[3].hist(design.loc[design["has_dip"], "dip_sigma_days"], bins=np.logspace(np.log10(0.25), np.log10(16), 35), color="tab:purple", alpha=0.75)
axes[3].set_xscale("log")
axes[3].set_title("Injected dip width parameter")
axes[3].set_xlabel("sigma-like days")
axes[4].hist(design["camera_offset_sigma_mag"], bins=35, color="tab:red", alpha=0.75)
axes[4].set_title("Camera offset scatter")
axes[4].set_xlabel("mag")
axes[5].hist(design["quasi_noise_amp_mag"], bins=35, color="teal", alpha=0.6)
axes[5].set_title("Quasi-correlated baseline imperfection")
axes[5].set_xlabel("mag")
fig.tight_layout()
fig.savefig(plots_dir / "simulation_population_audit.png", bbox_inches="tight")
plt.show()

class_counts = design["dip_class"].value_counts().rename_axis("dip_class").reset_index(name="n")
class_counts["fraction"] = class_counts["n"] / len(design)
display(class_counts)


## Production Threshold Comparison

These are the direct production-threshold results: LOO posterior probability uses the configured `SIGNIFICANCE_THRESHOLD`, and local log BF uses `LOGBF_THRESHOLD_DIP`. Read these as the operational comparison before considering threshold retuning.


In [ ]:
production = results[results["is_production"].fillna(False)].copy()
production_summary = run.summary_slices["production_overall"].sort_values(["mode", "trigger_family"])
display(production_summary)

print("Pairwise comparison of production triggers, matched by trial and baseline mode:")
display(run.pairwise_production)


In [ ]:
metric_cols = [
    "observable_recall",
    "precision_by_trial",
    "control_false_positive_rate",
    "off_target_detection_rate",
]
plot_df = production_summary.copy()
fig, axes = plt.subplots(len(metric_cols), 1, figsize=(13, 10), sharex=True)
for ax, metric in zip(axes, metric_cols):
    for family, sub in plot_df.groupby("trigger_family"):
        label = "LOO posterior" if family == "loo_posterior_prob" else "local log BF"
        ax.plot(sub["mode"], sub[metric], marker="o", label=label)
    ax.set_ylabel(metric)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
axes[0].legend()
axes[-1].tick_params(axis="x", rotation=35)
fig.suptitle("Production trigger performance by baseline mode", y=1.01)
fig.tight_layout()
fig.savefig(plots_dir / "production_trigger_comparison_by_mode.png", bbox_inches="tight")
plt.show()


## Threshold Sweeps

The next plots show how performance changes as the trigger threshold moves. The posterior-probability and log-BF x-axes are not the same physical scale, so compare their rate curves rather than their raw threshold positions.


In [ ]:
threshold_summary = run.summary_slices["all_thresholds_overall"].copy()
threshold_summary = threshold_summary.sort_values(["mode", "trigger_family", "threshold"])

metrics_to_plot = [
    ("observable_recall", "Observable recall"),
    ("precision_by_trial", "Precision by trial"),
    ("control_false_positive_rate", "Control false positive rate"),
    ("off_target_detection_rate", "Off-target detection rate"),
]
family_labels = {
    "loo_posterior_prob": "LOO posterior probability",
    "local_logbf": "local log BF",
}
family_colors = {
    "loo_posterior_prob": "goldenrod",
    "local_logbf": "tab:purple",
}

for mode in MODE_NAMES:
    sub_mode = threshold_summary[threshold_summary["mode"].eq(mode)].copy()
    if sub_mode.empty:
        continue
    fig, axes = plt.subplots(len(metrics_to_plot), 2, figsize=(14, 11), sharex="col")
    for row_idx, (metric, title) in enumerate(metrics_to_plot):
        for col_idx, family in enumerate(["loo_posterior_prob", "local_logbf"]):
            ax = axes[row_idx, col_idx]
            family_sub = sub_mode[sub_mode["trigger_family"].eq(family)].copy()
            ax.plot(family_sub["threshold"], family_sub[metric], marker="o", color=family_colors[family])
            ax.set_title(f"{title}: {family_labels[family]}")
            ax.set_ylabel("rate")
            ax.set_ylim(bottom=0)
            ax.grid(alpha=0.25)
            if row_idx == len(metrics_to_plot) - 1:
                ax.set_xlabel("posterior probability threshold" if family == "loo_posterior_prob" else "local log BF threshold")
    fig.suptitle(f"Threshold sweep: {mode}", y=1.01)
    fig.tight_layout()
    fig.savefig(plots_dir / f"threshold_sweep_{mode}.png", bbox_inches="tight")
    plt.show()


## Score-Space Relationship

Each row here is one scored light curve under one baseline mode. These plots show whether the strongest local log BF and strongest LOO posterior probability rank the same trials as event-like. Points near high log BF but low LOO probability are candidates where the local evidence spike is not supported by the posterior mixture model; points near high LOO probability but modest log BF are the reverse.


In [ ]:
score_ok = scores[scores["status"].eq("ok")].copy()
score_ok["truth_group"] = np.select(
    [
        ~score_ok["has_dip"].fillna(False).astype(bool),
        score_ok["truth_observable_actual"].fillna(False).astype(bool),
    ],
    ["control", "observable dip"],
    default="unobservable/sparse dip",
)
colors = {"control": "tab:gray", "observable dip": "tab:green", "unobservable/sparse dip": "tab:orange"}

for mode in MODE_NAMES:
    sub = score_ok[score_ok["mode"].eq(mode)].copy()
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 6))
    for group, group_sub in sub.groupby("truth_group"):
        ax.scatter(
            group_sub["max_log_bf_local"],
            group_sub["max_event_probability"],
            s=10,
            alpha=0.35,
            label=group,
            color=colors.get(group, None),
        )
    ax.axhline(posterior_probability_threshold(config.significance_threshold), color="goldenrod", lw=1.0, label="posterior production threshold")
    ax.axvline(config.logbf_threshold_dip, color="tab:purple", lw=1.0, label="logBF production threshold")
    ax.set_xlabel("max local log BF")
    ax.set_ylabel("max LOO P(event)")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"Score-space relationship: {mode}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(plots_dir / f"score_space_{mode}.png", bbox_inches="tight")
    plt.show()


## Production Slices

These tables break production-trigger behavior by injected dip amplitude, period, dip width, sampled point count, and periodic waveform. They are the most useful place to identify where one trigger family wins or fails.


In [ ]:
for name in [
    "production_by_amp",
    "production_by_period",
    "production_by_width",
    "production_by_points",
    "production_by_waveform",
]:
    print(f"\n=== {name} ===")
    display(run.summary_slices[name])


## Recovery Heatmaps

Heatmaps are shown separately for each production trigger family and baseline mode. Brighter cells mean higher target recovery among trials in that slice. The denominator is all trials in the slice, with `observable_recall` focused only on dips that had enough sampled truth support to be plausible detections.


In [ ]:
def heatmap_table(df, *, mode, family, metric="target_recovered", row="dip_amp_bin", col="period_bin"):
    sub = df[
        df["status"].eq("ok")
        & df["is_production"].fillna(False).astype(bool)
        & df["mode"].eq(mode)
        & df["trigger_family"].eq(family)
        & df["has_dip"].fillna(False).astype(bool)
        & df["truth_observable_actual"].fillna(False).astype(bool)
    ].copy()
    if sub.empty:
        return pd.DataFrame()
    table = sub.pivot_table(index=row, columns=col, values=metric, aggfunc=lambda x: np.mean(pd.Series(x).fillna(False).astype(bool)))
    return table

def plot_heatmap(table, ax, title):
    if table.empty:
        ax.set_axis_off()
        ax.set_title(title + " (no data)")
        return
    im = ax.imshow(table.to_numpy(dtype=float), aspect="auto", vmin=0, vmax=np.nanmax(table.to_numpy(dtype=float)) if np.nanmax(table.to_numpy(dtype=float)) > 0 else 1)
    ax.set_xticks(np.arange(table.shape[1]), table.columns, rotation=35, ha="right")
    ax.set_yticks(np.arange(table.shape[0]), table.index)
    ax.set_title(title)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            val = table.iloc[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7, color="white" if val > 0.5 * np.nanmax(table.to_numpy(dtype=float)) else "black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for mode in MODE_NAMES:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for ax, family in zip(axes, ["loo_posterior_prob", "local_logbf"]):
        table = heatmap_table(results, mode=mode, family=family)
        label = "LOO posterior" if family == "loo_posterior_prob" else "local log BF"
        plot_heatmap(table, ax, f"Observable recovery: {label}\n{mode}")
    fig.tight_layout()
    fig.savefig(plots_dir / f"recovery_heatmap_{mode}.png", bbox_inches="tight")
    plt.show()


## Direct Disagreement Tables

These tables are matched at the trial and baseline-mode level for the two production triggers. They isolate cases where one trigger recovers the injected dip and the other does not, plus control false positives unique to either trigger.


In [ ]:
def paired_production_table(df, mode):
    prod = df[df["status"].eq("ok") & df["is_production"].fillna(False).astype(bool) & df["mode"].eq(mode)].copy()
    pp = prod[prod["trigger_family"].eq("loo_posterior_prob")].set_index("trial_id")
    bf = prod[prod["trigger_family"].eq("local_logbf")].set_index("trial_id")
    common = pp.index.intersection(bf.index)
    if common.empty:
        return pd.DataFrame()
    cols = [
        "dip_class", "dip_amp_mag", "dip_sigma_days", "truth_support_points_actual", "truth_peak_snr_actual",
        "period_days", "n_points_actual", "waveform_kind", "max_event_probability", "max_log_bf_local",
        "baseline_mae_outside_dip", "resid_rms_outside_dip",
    ]
    out = pp.loc[common, cols].copy()
    out["posterior_recovered"] = pp.loc[common, "target_recovered"].fillna(False).astype(bool)
    out["logbf_recovered"] = bf.loc[common, "target_recovered"].fillna(False).astype(bool)
    out["posterior_fp"] = pp.loc[common, "false_positive"].fillna(False).astype(bool)
    out["logbf_fp"] = bf.loc[common, "false_positive"].fillna(False).astype(bool)
    out["posterior_event_points"] = pp.loc[common, "event_points"]
    out["logbf_event_points"] = bf.loc[common, "event_points"]
    out["posterior_off_target"] = pp.loc[common, "off_target_detection"].fillna(False).astype(bool)
    out["logbf_off_target"] = bf.loc[common, "off_target_detection"].fillna(False).astype(bool)
    return out.reset_index()

paired_tables = {mode: paired_production_table(results, mode) for mode in MODE_NAMES}

for mode, table in paired_tables.items():
    if table.empty:
        continue
    print(f"\n=== {mode}: posterior-only recovered ===")
    display(table[table["posterior_recovered"] & ~table["logbf_recovered"]].sort_values("max_event_probability", ascending=False).head(20))
    print(f"\n=== {mode}: logBF-only recovered ===")
    display(table[~table["posterior_recovered"] & table["logbf_recovered"]].sort_values("max_log_bf_local", ascending=False).head(20))
    print(f"\n=== {mode}: posterior-only control false positives ===")
    display(table[table["posterior_fp"] & ~table["logbf_fp"]].sort_values("max_event_probability", ascending=False).head(20))
    print(f"\n=== {mode}: logBF-only control false positives ===")
    display(table[~table["posterior_fp"] & table["logbf_fp"]].sort_values("max_log_bf_local", ascending=False).head(20))


## Off-Target Detections and False Positives

These are detections that pass the trigger and run-gating thresholds but do not overlap the injected dip support. In dip-bearing trials they are off-target detections; in control trials they are false positives. High counts here usually mean the trigger is responding to periodic-template imperfections, camera offsets, seasonal gaps, or isolated noisy points rather than the injected one-off event.


In [ ]:
for family in ["loo_posterior_prob", "local_logbf"]:
    label = "LOO posterior" if family == "loo_posterior_prob" else "local log BF"
    sub = production[production["trigger_family"].eq(family)].copy()
    print(f"\n=== {label}: largest off-target detections ===")
    display(
        sub[sub["off_target_detection"].fillna(False)]
        .sort_values(["max_log_bf_local", "max_event_probability"], ascending=False)
        .head(30)[[
            "trial_id", "mode", "dip_class", "dip_amp_mag", "truth_support_points_actual", "truth_peak_snr_actual",
            "period_days", "n_points_actual", "waveform_kind", "max_event_probability", "max_log_bf_local",
            "event_points", "raw_trigger_points", "baseline_mae_outside_dip", "resid_rms_outside_dip",
        ]]
    )
    print(f"\n=== {label}: largest control false positives ===")
    display(
        sub[sub["false_positive"].fillna(False)]
        .sort_values(["max_log_bf_local", "max_event_probability"], ascending=False)
        .head(30)[[
            "trial_id", "mode", "period_days", "n_points_actual", "waveform_kind", "max_event_probability", "max_log_bf_local",
            "event_points", "raw_trigger_points", "baseline_mae_outside_dip", "resid_rms_outside_dip",
        ]]
    )


## Diagnostic Example Plots

For the selected mode, the notebook picks representative production-trigger disagreements and recomputes the simulated light curve. The plots overlay truth support, posterior-triggered points, logBF-triggered points, per-point LOO probability, per-point log BF, and the folded baseline.


In [ ]:
EXAMPLE_MODE = os.environ.get("MALCA_TRIGGER_MODE_EXAMPLE_MODE", "phase_template_true_period")
examples = select_disagreement_trials(results, mode=EXAMPLE_MODE)
print(EXAMPLE_MODE)
print(examples)

for label, trial_id in examples.items():
    if trial_id is None:
        continue
    fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=False)
    plot_trigger_mode_trial_diagnostic(run, int(trial_id), mode=EXAMPLE_MODE, ax=axes)
    fig.suptitle(f"{label}: trial {trial_id}", y=1.01)
    fig.tight_layout()
    fig.savefig(plots_dir / f"diagnostic_{EXAMPLE_MODE}_{label}_trial_{trial_id}.png", bbox_inches="tight")
    plt.show()


## Quick Interpretation Checklist

Use this notebook in this order:

1. Check the simulation audit to make sure the synthetic population is broad enough for the question being asked.
2. Read `production_overall` and `pairwise_production` first. This is the current operational LOO-vs-BF comparison.
3. Use threshold sweeps to decide whether a trigger family is fundamentally weak or merely poorly thresholded.
4. Use slice tables and recovery heatmaps to identify the regimes where the trigger fails: low amplitude, sparse sampling, long/short periods, broad dips, or waveform-specific baseline mismatch.
5. Inspect disagreement plots. The top panels tell you whether the baseline is wrong; the posterior and logBF panels tell you whether the trigger family is suppressing or amplifying the same residual excursions.

The most actionable patterns are:

- **High logBF-only recovery with many logBF false positives**: local evidence is sensitive but needs stricter run/morphology gating or a higher threshold.
- **Posterior-only recovery with low false positives**: LOO probability is finding coherent event support that local logBF thresholding misses.
- **Both triggers miss high-SNR observable dips**: the bottleneck is probably baseline construction, run gating, injected event support definition, or the event-grid parameterization rather than trigger mode.
- **Both triggers fire off target**: baseline residual structure is being scored as a dip; inspect camera offsets, phase-template period errors, and seasonal gaps.
